In [13]:
import random
import nltk
from nltk import find
import torchaudio
import torch
from transformers import AutoConfig, AutoModel
import os
from gensim.models import Word2Vec, KeyedVectors
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = "cpu"
print('Device available is', device)

seed = 7 
random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)


Device available is cuda


In [2]:
# function to generate audio embeddings

MODEL_NAME = "facebook/wav2vec2-base"
encoder_config = AutoConfig.from_pretrained(MODEL_NAME)
encoder = AutoModel.from_pretrained(MODEL_NAME, device_map="cuda")

def generate_embedding(filename, beginning, end):
  # make sure we're only generating an embedding for the specific word
  clip_start = int(beginning * 16000)
  clip_end = int(end * 16000)

  num_frames = clip_end - clip_start

  try:
    waveform,sample_rate=torchaudio.load(filename, frame_offset=clip_start, num_frames=num_frames)
    #ensures both input and encoder are on the same device
    waveform = waveform.to(device)
    # Stereo to mono
    if waveform.shape[0] > 1:
      waveform = torch.mean(waveform,dim=0,keepdim=True)
    # Resample
    if sample_rate != 16000:
      waveform = torchaudio.functional.sample(
          waveform,
          sample_rate,
          16000,
      )
    # extract features
    emission = encoder(waveform.to(device)).last_hidden_state
    emission = emission.detach().cpu().numpy()
    return emission
  except Exception as e:
    print(f"Error processing {filename}: {e}")
    return None

Loading weights: 100%|██████████| 211/211 [00:00<00:00, 1892.92it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_q.bias               | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
# load the transcript data and tokenize sentences
data_path = "LibriSpeech/dev-clean/"

sentences = []
vocab_embeddings = {}
audio_paths = [] 


# load sentences
# for reader_id in os.listdir(data_path):
# for reader_idx in range(5):
    # reader_id = os.listdir(data_path)[reader_idx]
for reader_id in tqdm(os.listdir(data_path), desc="Reader", total=len(os.listdir(data_path))):
    reader_path = data_path + str(reader_id) + "/"

    for chapter_id in os.listdir(reader_path):
    # for chapter_id in tqdm(os.listdir(reader_path), desc="Chapter ", total=len(os.listdir(reader_path))):
        chapter_file_path = reader_path + str(chapter_id) + "/"
        transcript_file = str(reader_id) + "-" + str(chapter_id) + ".alignment.txt"

        with open(chapter_file_path + transcript_file) as file:
            for line in file.readlines():
            # for line in tqdm(file.readlines(), desc="Generating for " + chapter_file_path + transcript_file, total=len(file.readlines())):
                # PER SENTENCE LEVEL HERE !
                audio_filename, words, timings = line.split()

                audio_path = chapter_file_path + audio_filename + ".flac"

                words = words.split(",")[1:-1] # first and last entries are always silences, so we remove these
                words = [word.lower() for word in words]
                
                timings = timings.replace('"', "") # remove double quotes
                timings = timings.split(",")
                timings = [float(timing) for timing in timings] # surely there is a better way to do this, but this works for now

                for i in range(len(words)):

                    if words[i] not in vocab_embeddings.keys():
                        vocab_embeddings[words[i]] = []
                    
                    # i+1 and i+2 since the first timing is the end of the silence, and the last one is the ending of the clip-ending silence
                    vocab_embeddings[words[i]].append(generate_embedding(audio_path, timings[i + 1], timings[i + 2]))
                
                sentences.append(words)
            
        # for path in os.listdir(chapter_file_path):
        #     if path.endswith(".flac"):
        #         audio_paths.append(path)

# print(audio_paths)


Generating for LibriSpeech/dev-clean/1272/128104/1272-128104.alignment.txt: 15it [00:05,  2.59it/s]


Error processing LibriSpeech/dev-clean/1272/135031/1272-135031-0009.flac: Calculated padded input size per channel: (1). Kernel size: (2). Kernel size can't be greater than actual input size


Generating for LibriSpeech/dev-clean/1272/135031/1272-135031.alignment.txt: 25it [00:05,  4.87it/s]
Generating for LibriSpeech/dev-clean/1272/141231/1272-141231.alignment.txt: 33it [00:06,  4.80it/s]
Generating for LibriSpeech/dev-clean/1462/170138/1462-170138.alignment.txt: 28it [00:07,  4.00it/s]
Generating for LibriSpeech/dev-clean/1462/170142/1462-170142.alignment.txt: 43it [00:07,  5.58it/s]
Generating for LibriSpeech/dev-clean/1462/170145/1462-170145.alignment.txt: 23it [00:04,  5.28it/s]
Generating for LibriSpeech/dev-clean/1673/143396/1673-143396.alignment.txt: 21it [00:11,  1.78it/s]
Generating for LibriSpeech/dev-clean/1673/143397/1673-143397.alignment.txt: 21it [00:08,  2.51it/s]
Generating for LibriSpeech/dev-clean/174/168635/174-168635.alignment.txt: 23it [00:06,  3.59it/s]
Generating for LibriSpeech/dev-clean/174/50561/174-50561.alignment.txt: 20it [00:05,  3.65it/s]
Generating for LibriSpeech/dev-clean/174/84280/174-84280.alignment.txt: 16it [00:05,  3.04it/s]
Generating

Error processing LibriSpeech/dev-clean/1988/24833/1988-24833-0011.flac: Calculated padded input size per channel: (1). Kernel size: (2). Kernel size can't be greater than actual input size


Generating for LibriSpeech/dev-clean/1988/24833/1988-24833.alignment.txt: 29it [00:06,  4.63it/s]
Generating for LibriSpeech/dev-clean/1993/147149/1993-147149.alignment.txt: 31it [00:11,  2.59it/s]
Generating for LibriSpeech/dev-clean/1993/147964/1993-147964.alignment.txt: 11it [00:03,  3.39it/s]
Generating for LibriSpeech/dev-clean/1993/147965/1993-147965.alignment.txt: 9it [00:01,  4.53it/s]
Generating for LibriSpeech/dev-clean/1993/147966/1993-147966.alignment.txt: 7it [00:01,  4.65it/s]
Generating for LibriSpeech/dev-clean/2035/147960/2035-147960.alignment.txt: 17it [00:03,  5.41it/s]
Generating for LibriSpeech/dev-clean/2035/147961/2035-147961.alignment.txt: 41it [00:06,  6.18it/s]
Generating for LibriSpeech/dev-clean/2035/152373/2035-152373.alignment.txt: 19it [00:06,  2.84it/s]
Generating for LibriSpeech/dev-clean/2078/142845/2078-142845.alignment.txt: 52it [00:14,  3.58it/s]
Generating for LibriSpeech/dev-clean/2086/149214/2086-149214.alignment.txt: 5it [00:02,  2.30it/s]
Gener

...

Generating for LibriSpeech/dev-clean/3853/163249/3853-163249.alignment.txt: 57it [00:18,  3.15it/s]
Generating for LibriSpeech/dev-clean/422/122949/422-122949.alignment.txt: 36it [00:16,  2.13it/s]
Generating for LibriSpeech/dev-clean/5338/24615/5338-24615.alignment.txt: 15it [00:05,  2.51it/s]
Generating for LibriSpeech/dev-clean/5338/24640/5338-24640.alignment.txt: 10it [00:03,  2.80it/s]


Error processing LibriSpeech/dev-clean/5338/284437/5338-284437-0018.flac: Calculated padded input size per channel: (1). Kernel size: (2). Kernel size can't be greater than actual input size


Generating for LibriSpeech/dev-clean/5338/284437/5338-284437.alignment.txt: 34it [00:07,  4.62it/s]
Generating for LibriSpeech/dev-clean/5536/43358/5536-43358.alignment.txt: 20it [00:05,  3.48it/s]
Generating for LibriSpeech/dev-clean/5536/43359/5536-43359.alignment.txt: 19it [00:05,  3.61it/s]
Generating for LibriSpeech/dev-clean/5536/43363/5536-43363.alignment.txt: 20it [00:06,  2.93it/s]
Generating for LibriSpeech/dev-clean/5694/64025/5694-64025.alignment.txt: 24it [00:05,  4.16it/s]
Generating for LibriSpeech/dev-clean/5694/64029/5694-64029.alignment.txt: 33it [00:06,  5.07it/s]
Generating for LibriSpeech/dev-clean/5694/64038/5694-64038.alignment.txt: 26it [00:05,  4.99it/s]
Generating for LibriSpeech/dev-clean/5895/34615/5895-34615.alignment.txt: 22it [00:04,  5.16it/s]
Generating for LibriSpeech/dev-clean/5895/34622/5895-34622.alignment.txt: 24it [00:04,  4.91it/s]
Generating for LibriSpeech/dev-clean/5895/34629/5895-34629.alignment.txt: 34it [00:05,  5.71it/s]
Generating for Lib

Error processing LibriSpeech/dev-clean/777/126732/777-126732-0024.flac: Calculated padded input size per channel: (1). Kernel size: (2). Kernel size can't be greater than actual input size


Generating for LibriSpeech/dev-clean/777/126732/777-126732.alignment.txt: 82it [00:17,  4.56it/s]
Generating for LibriSpeech/dev-clean/7850/111771/7850-111771.alignment.txt: 10it [00:02,  4.42it/s]
Generating for LibriSpeech/dev-clean/7850/281318/7850-281318.alignment.txt: 24it [00:04,  5.32it/s]
Generating for LibriSpeech/dev-clean/7850/286674/7850-286674.alignment.txt: 18it [00:03,  4.65it/s]
Generating for LibriSpeech/dev-clean/7850/73752/7850-73752.alignment.txt: 20it [00:05,  3.56it/s]
Generating for LibriSpeech/dev-clean/7976/105575/7976-105575.alignment.txt: 30it [00:05,  5.02it/s]
Generating for LibriSpeech/dev-clean/7976/110124/7976-110124.alignment.txt: 26it [00:04,  5.24it/s]
Generating for LibriSpeech/dev-clean/7976/110523/7976-110523.alignment.txt: 22it [00:06,  3.15it/s]
Generating for LibriSpeech/dev-clean/8297/275154/8297-275154.alignment.txt: 28it [00:05,  5.59it/s]
Generating for LibriSpeech/dev-clean/8297/275155/8297-275155.alignment.txt: 33it [00:06,  5.39it/s]
Gene

In [17]:
# build Word2Vec models
librispeech_model = Word2Vec(sentences)
# librispeech_model.build_vocab(sentences)
print("librispeech Word2Vec model trained from " + str(len(sentences)) + " sentences")

# word2vec model
nltk.download('word2vec_sample')
word2vec_sample = str(find('models/word2vec_sample/pruned.word2vec.txt'))
word2vec_model = KeyedVectors.load_word2vec_format(word2vec_sample, binary=False)

librispeech Word2Vec model trained from 2703 sentences


[nltk_data] Downloading package word2vec_sample to
[nltk_data]     C:\Users\dunna\AppData\Roaming\nltk_data...
[nltk_data]   Package word2vec_sample is already up-to-date!


In [21]:
# text embedding comparisons 
# for key in vocab_embeddings.keys():
#     print(key ,vocab_embeddings[key][0])

libri_word2vec_keys = set(librispeech_model.wv.index_to_key)
word2vec_keys = set(word2vec_model.index_to_key)
wav2vec_keys = set(vocab_embeddings.keys())

print("Number of word2vec keys: ", len(word2vec_keys))
print("Number of wav2vec keys: ", len(wav2vec_keys))
print(len(word2vec_keys.symmetric_difference(wav2vec_keys)))


test_words = [random.choice(librispeech_model.wv.index_to_key) for i in range(5)]

for word1 in test_words:
    for word2 in test_words:
        # print(word1, word2)
        libriSim = librispeech_model.wv.similarity(word1, word2)
        brown_sim = word2vec_model.similarity(word1, word2)

        print("word2vec similarity for " + word1 + " and " + word2 + "\t : " + str(brown_sim))
        print("Libri similarity for " + word1 + " and " + word2 + "\t : " + str(libriSim))


# test2 = brown_model.wv.similarity("watch", "look")
# print(test2)

Number of word2vec keys:  43981
Number of wav2vec keys:  8334
38991
word2vec similarity for created and created	 : 1.0
Libri similarity for created and created	 : 0.9999999
word2vec similarity for created and age	 : -0.030624654
Libri similarity for created and age	 : 0.9921317
word2vec similarity for created and like	 : 0.18283369
Libri similarity for created and like	 : 0.9936553
word2vec similarity for created and half	 : 0.07650673
Libri similarity for created and half	 : 0.9932871
word2vec similarity for created and change	 : 0.09049655
Libri similarity for created and change	 : 0.98861784
word2vec similarity for age and created	 : -0.030624654
Libri similarity for age and created	 : 0.9921317
word2vec similarity for age and age	 : 0.99999994
Libri similarity for age and age	 : 1.0
word2vec similarity for age and like	 : 0.053234175
Libri similarity for age and like	 : 0.9984351
word2vec similarity for age and half	 : 0.049303368
Libri similarity for age and half	 : 0.9983473
word

In [ ]:
# audio embedding comparisons




AttributeError: 'KeyedVectors' object has no attribute 'wv'

In [ ]:
# check to see the cosine similarity of 